# Phase 4: Causal Patching — Activation Patching

**Experimental protocol (Denoising):**
1. Run **clean prompt** → save clean hidden states for all layers (at the last token of `<hop1>`)
2. Run **counterfactual (corrupted) prompt** normally → save the baseline (hallucinated) answer
3. For **each layer**: Run **counterfactual prompt again**, but REPLACE the layer's hidden state (at the last token of `<hop1>`) with the CLEAN state harvested in step 1
4. Measure whether injecting the clean state successfully restores the correct answer

If the correct answer is restored in ≥50% of cases for a given layer, that layer is causally responsible for the hop-1 reasoning step.

In [ ]:
!pip install -q transformers accelerate huggingface_hub
import huggingface_hub
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    huggingface_hub.login(token=hf_token)
    print("Logged in via Kaggle Secrets!")
except Exception as e:
    print(f"Kaggle Secrets failed ({e}), using fallback...")
    hf_token = "REPLACE_ME_WITH_YOUR_RAW_HF_TOKEN"
    huggingface_hub.login(token=hf_token)

In [ ]:
import torch
import numpy as np
import random

# Set seed for determinism
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

import json, torch
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"

print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)
model.eval()
print(f"Model loaded. Device: {model.device}")

In [ ]:
print("Downloading dataset...")
file_path = hf_hub_download(
    repo_id="AnishRacherla/LinguaFranca-Phase3",
    filename="data/2wikimultihopqa/augmented.jsonl",
    repo_type="dataset"
)
with open(file_path) as f:
    data = [json.loads(line) for line in f]

counterfactuals = [d for d in data if d.get("is_counterfactual", False)]
cleans = {d["id"]: d for d in data if not d.get("is_counterfactual", False)}
print(f"Loaded {len(cleans)} clean examples, {len(counterfactuals)} counterfactuals.")

# Build minimal pairs: clean example answered correctly (hop1 label=0) 
# AND its CF twin answered incorrectly (hop1 label=1)
pairs = []
for cf in counterfactuals:
    clean = cleans.get(cf.get("clean_pair_id"))
    if not clean:
        continue
    clean_h1 = next((h for h in clean.get("hops", []) if h["hop_idx"] == 1), None)
    cf_h1 = next((h for h in cf.get("hops", []) if h["hop_idx"] == 1), None)
    if clean_h1 and cf_h1 and clean_h1.get("label") == 0 and cf_h1.get("label") == 1:
        pairs.append((clean, cf))
            

print(f"Found {len(pairs)} perfect minimal pairs.")

In [ ]:
# ===== CELL 4 (replace the old one) — logit-based causal tracing =====
# Keeps cells 1-3 as they are (model, tokenizer, `pairs` must already exist).
# No generation, no sampling: one forward pass per patch -> deterministic and fast.

import re, json
import numpy as np
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

num_layers = model.config.num_hidden_layers
WINDOW = 5        # window patching: layers l-2 .. l+2
MIN_GAP = 1.0     # pair must show >= 1 nat clean-vs-corrupted gap, else it is not a valid test

def get_hop1(cot):
    m = re.search(r'(<hop1>.*?</hop1>)', cot, re.DOTALL)
    return m.group(1) if m else ""

def enc(text):
    return tokenizer(text, add_special_tokens=False, return_tensors="pt").input_ids[0]

def make_harvest_hook(store, l):
    def hook(module, args, output):
        hidden = output[0] if isinstance(output, tuple) else output
        store[l] = hidden[0].detach().clone()          # [seq, d] for this layer
        return output
    return hook

def make_patch_hook(positions, vecs):
    def hook(module, args, output):
        hidden = output[0] if isinstance(output, tuple) else output
        hidden = hidden.clone()
        hidden[0, positions, :] = vecs
        if isinstance(output, tuple):
            return (hidden,) + tuple(output[1:])
        return hidden
    return hook

@torch.no_grad()
def target_logprob(input_ids, P, mask, patches=None):
    """Sum log P(target tokens) where target = input_ids[P:], restricted to `mask`.
    patches: {layer: (positions, clean_vectors)}"""
    handles = []
    if patches:
        for l, (pos, vecs) in patches.items():
            handles.append(model.model.layers[l].register_forward_hook(make_patch_hook(pos, vecs)))
    try:
        logits = model(input_ids.unsqueeze(0).to(model.device)).logits[0].float()
    finally:
        for h in handles:
            h.remove()
    tgt = input_ids[P:].to(model.device)
    lp = F.log_softmax(logits[P - 1:-1], dim=-1).gather(1, tgt.unsqueeze(1)).squeeze(1)
    return lp[mask.to(model.device)].sum().item()

CONDITIONS = ["entity_single", "entity_window", "last_single"]
results, skipped = [], {"no_hop1": 0, "len_mismatch": 0, "no_diff": 0, "no_gap": 0}

# ONLY TEST FIRST 10 PAIRS AS REQUESTED
pairs_subset = pairs[:10]

for clean, cf in tqdm(pairs_subset, desc="Causal tracing"):
    tgt_text = get_hop1(clean["generated_cot"])            # the CORRECT hop-1 the clean run produced
    if not tgt_text:
        skipped["no_hop1"] += 1; continue

    c_ids, f_ids, t_ids = enc(clean["prompt"]), enc(cf["prompt"]), enc(tgt_text)
    if len(c_ids) != len(f_ids):                           # token-length-matched swaps should never hit this
        skipped["len_mismatch"] += 1; continue
    ent_pos = (c_ids != f_ids).nonzero().flatten()        # exactly the swapped-entity tokens
    if len(ent_pos) == 0:
        skipped["no_diff"] += 1; continue

    P = len(c_ids)
    clean_in = torch.cat([c_ids, t_ids])
    cf_in = torch.cat([f_ids, t_ids])                      # same target, corrupted prompt

    # Score only the reasoning part of hop 1 (e.g. the director's name), not the copied film-name tokens
    ent_tok_ids = torch.cat([c_ids[ent_pos], f_ids[ent_pos]])
    mask = ~torch.isin(t_ids, ent_tok_ids)
    if mask.sum() == 0:
        skipped["no_gap"] += 1; continue

    # 1) clean run: harvest every layer's hidden states
    store = {}
    hs = [model.model.layers[l].register_forward_hook(make_harvest_hook(store, l)) for l in range(num_layers)]
    try:
        lp_clean = target_logprob(clean_in, P, mask)
    finally:
        for h in hs:
            h.remove()

    # 2) corrupted run, no patch
    lp_cf = target_logprob(cf_in, P, mask)
    gap = lp_clean - lp_cf
    if gap < MIN_GAP:                                      # model doesn't actually behave differently -> invalid pair
        skipped["no_gap"] += 1; continue

    # 3) patched corrupted runs
    ent_dev = ent_pos.to(model.device)
    last_dev = torch.tensor([P - 1], device=model.device)
    rec = {k: [] for k in CONDITIONS}
    for l in range(num_layers):
        win = range(max(0, l - WINDOW // 2), min(num_layers, l + WINDOW // 2 + 1))
        for key, pos, layers in [("entity_single", ent_dev, [l]),
                                 ("entity_window", ent_dev, win),
                                 ("last_single", last_dev, [l])]:
            patches = {k: (pos, store[k][pos]) for k in layers}
            lp = target_logprob(cf_in, P, mask, patches)
            rec[key].append((lp - lp_cf) / gap)            # 0 = no effect, 1 = fully restored
    rec["gap"] = gap
    results.append(rec)
    del store
    torch.cuda.empty_cache()

# ===== Results =====
n = len(results)
print(f"\nPairs: {len(pairs_subset)} | valid: {n} | skipped: {skipped}\n")
if n > 0:
    R ={k: np.array([r[k] for r in results]) for k in CONDITIONS}

    for k in CONDITIONS:
        mean = R[k].mean(0)
        ci = 1.96 * R[k].std(0) / np.sqrt(n)
        frac = (R[k] > 0.5).mean(0) * 100
        print(f"--- {k} ---")
        for l in range(num_layers):
            print(f"Layer {l:02d}: recovery {mean[l]:5.2f} ± {ci[l]:.2f} | pairs >50% restored: {frac[l]:5.1f}%")
        print()

    import matplotlib.pyplot as plt
    plt.figure(figsize=(10, 5))
    for k in CONDITIONS:
        m = R[k].mean(0); c = 1.96 * R[k].std(0) / np.sqrt(n)
        plt.plot(range(num_layers), m, marker="o", label=k)
        plt.fill_between(range(num_layers), m - c, m + c, alpha=0.2)
    plt.axhline(0, color="gray", lw=0.8)
    plt.xlabel("Layer"); plt.ylabel("Fraction of clean-vs-corrupted gap recovered")
    plt.title(f"Causal tracing of hop-1 (n={n} pairs)")
    plt.legend(); plt.grid(alpha=0.3); plt.show()

    with open("causal_trace_results.json", "w") as f:
        json.dump({"skipped": skipped, "n": n,
                   "results": [{k: (v if k == "gap" else list(map(float, v))) for k, v in r.items()} for r in results]}, f)
    print("Saved causal_trace_results.json")
else:
    print("No valid pairs — check the skipped counts above before going further.")
